# Этап 2: Расчет баллов и весов

Задача: расчет баллов и весов и сверка с model_data.json (Витрина проекта)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

#BASE = Path(r"C:\Users\dak\OneDrive\Документы\3 Горнолыжные курорты\Travel_Designer_Resort_Ranking")
BASE = Path (".")

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"


## Загрузка витрины - эталона

In [2]:
with open(PROCESSED / "model_data.json", encoding="utf-8-sig") as f:
    model_data = json.load(f)

model_df = pd.DataFrame(model_data["model_rows"])
weights = model_data["weights"]

print("Веса:", weights)
print("Курортов:", len(model_df))

Веса: {'region_trips': 25.0, 'region_growth': 10.0, 'avg_stay': 5.0, 'pistes': 10.0, 'lifts': 7.0, 'lodging': 5.0, 'amenities': 3.0, 'snow_days': 8.0, 'freezing_days': 6.0, 'snowfall': 6.0, 'airport': 8.0, 'transport': 2.0, 'price': 5.0}
,Курортов: 32


In [3]:
model_df = pd.DataFrame(model_data["model_rows"])
weights = model_data["weights"]

rename_map = {
    "resort": "Курорт",
    "region": "Регион",
    "region_trips": "Поездки по региону",
    "region_growth": "Рост турпотока",
    "avg_stay": "Средняя длительность, ночей",
    "pistes": "Трассы, км",
    "lifts": "Подъёмники, шт.",
    "lodging": "Размещение, мест",
    "amenities": "Инфраструктура",
    "snow_days": "Снежных дней",
    "freezing_days": "Дней с морозом",
    "snowfall": "Снегопад, см",
    "airport": "Аэропорт, км",
    "transport": "Транспорт",
    "price": "Цена, ₽",
    "score": "Итоговый балл",
}

model_df = model_df.rename(columns=rename_map)

weights_ru = (
    pd.Series(weights, name="Вес, %")
      .rename(index=rename_map)
      .sort_values(ascending=False)
)

display(weights_ru)

Поездки по региону             25.0
Рост турпотока                 10.0
Трассы, км                     10.0
Снежных дней                    8.0
Аэропорт, км                    8.0
Подъёмники, шт.                 7.0
Дней с морозом                  6.0
Снегопад, см                    6.0
Средняя длительность, ночей     5.0
Размещение, мест                5.0
Цена, ₽                         5.0
Инфраструктура                  3.0
Транспорт                       2.0
Name: Вес, %, dtype: float64

## Нормализация

In [4]:
def min_max_score(s):
    """Min-max: 100 * (x - min) / (max - min)"""
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(50.0, index=s.index) # все равны — ставим 50
    return 100 * (s - lo) / (hi - lo)

def log_min_max_score(s):
    """log-min-max с 1+x, чтобы не ловить ln(0)"""
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.min(), s.max()
    ln = np.log1p(s)
    if ln.max() == ln.min():
        return pd.Series(50.0, index=s.index)
    return 100 * (ln - ln.min()) / (ln.max() - ln.min())

def inverse_score(s):
    """Обратная шкала: меньше — лучше"""
    return 100 - log_min_max_score(s)

def growth_score(s):
    """clip(50 + 200*growth, 0, 100)"""
    return (50 + 200 * s).clip(0, 100)

def stay_score(s):
    """clip((avg_stay - 1) / 6 * 100, 0, 100)"""
    return ((s - 1) / 6 * 100).clip(0, 100)

In [9]:
# региональный спрос
model_df["region_trips_score_calc"] = log_min_max_score(model_df["trips_2025"])
model_df["region_growth_score_calc"] = growth_score(model_df["trips_growth_2025"])
model_df["avg_stay_score_calc"] = stay_score(model_df["avg_stay_nights_2025"])

# инфраструктура
model_df["pistes_score_calc"] = log_min_max_score(model_df["piste_segments_6km"])
model_df["lifts_score_calc"] = log_min_max_score(model_df["lift_lines_6km"])
model_df["lodging_score_calc"] = log_min_max_score(model_df["lodging_objects_6km"])
model_df["amenities_score_calc"] = log_min_max_score(model_df["food_objects_6km"] + model_df["attraction_objects_6km"])

# зимние условия
model_df["snow_days_score_calc"] = min_max_score(model_df["avg_snow_days_per_season"])
model_df["freezing_days_score_calc"] = min_max_score(model_df["avg_freezing_days_per_season"])
model_df["snowfall_score_calc"] = log_min_max_score(model_df["avg_snowfall_cm_per_season"])

# доступность
model_df["airport_score_calc"] = inverse_score(model_df["nearest_airport_distance_km"])
model_df["transport_score_calc"] = log_min_max_score(model_df["transport_objects_6km"])

# цена (missing price -> 50, как в ТЗ)
price = model_df["adult_skipass_avg_rub"].copy()
price_filled = price.fillna(price.mean())
model_df["price_score_calc"] = inverse_score(price_filled)
model_df.loc[price.isna(), "price_score_calc"] = 50.0

In [6]:

weights = model_data["weights"]

model_df["popularity_score_calc"] = (
    model_df["region_trips_score_calc"] * weights["region_trips"]
    + model_df["region_growth_score_calc"] * weights["region_growth"]
    + model_df["avg_stay_score_calc"] * weights["avg_stay"]
    + model_df["pistes_score_calc"] * weights["pistes"]
    + model_df["lifts_score_calc"] * weights["lifts"]
    + model_df["lodging_score_calc"] * weights["lodging"]
    + model_df["amenities_score_calc"] * weights["amenities"]
    + model_df["snow_days_score_calc"] * weights["snow_days"]
    + model_df["freezing_days_score_calc"] * weights["freezing_days"]
    + model_df["snowfall_score_calc"] * weights["snowfall"]
    + model_df["airport_score_calc"] * weights["airport"]
    + model_df["transport_score_calc"] * weights["transport"]
    + model_df["price_score_calc"] * weights["price"]
) / 100

print(model_df[["name_ru", "popularity_score_default", "popularity_score_calc"]].head(10))

          name_ru  popularity_score_default  popularity_score_calc
,0  Газпром Поляна                 73.788787              73.788787
,1  Красная Поляна                 71.687850              71.687850
,2      Роза Хутор                 71.569843              71.569843
,3        Сорочаны                 70.143513              70.143513
,4           Архыз                 59.250873              59.250873
,5          Домбай                 57.315487              57.315487
,6           Игора                 55.931923              55.931923
,7         Шерегеш                 55.657061              55.657061
,8   Горный воздух                 52.786718              52.786718
,9    Бобровый лог                 51.092041              51.092041


## Сравнение с эталоном

In [7]:
diff = (model_df["popularity_score_calc"] - model_df["popularity_score_default"]).abs()

print(f"Максимальное расхождение: {diff.max():.4f}")
print(f"Среднее расхождение: {diff.mean():.4f}")
print(f"Строк с расхождением >0.2: {(diff > 0.2).sum()}")

# где большие расхождения
bad = model_df.loc[diff > 0.2, ["resort_id", "name_ru",
                                 "popularity_score_default", "popularity_score_calc"]]
print(bad.to_string(index=False))

Максимальное расхождение: 0.0000
,Среднее расхождение: 0.0000
,Строк с расхождением >0.2: 0
,Empty DataFrame
,Columns: [resort_id, name_ru, popularity_score_default, popularity_score_calc]
,Index: []


Сгенерировала свои баллы — через log_min_max_score, growth_score, stay_score, inverse_score.

Сравнила с эталоном — через вычитание.

Получила 0.0 — значит, мой расчёт совпал с расчётом заказчика.